In [8]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import _tree

In [9]:
# ============================================================
# USER SETTINGS
# ============================================================
DATASET_PATH = "dataset.csv"
EXPORT_NUM_TREES = 7          # <<< YOU CONTROL THIS
CPP_OUTPUT_FILE = "random_forest_model.cpp"
RANDOM_STATE = 42
# ============================================================


# ============================================================
# LOAD + PREPROCESS DATA
# ============================================================
data = pd.read_csv(DATASET_PATH)

# Drop timestamp columns
data = data.loc[:, ~data.columns.str.contains("timestamp", case=False)]

# Shuffle
data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X = data.drop(columns=["label"])
y_raw = data["label"]

# Encode labels
y, label_mapping = pd.factorize(y_raw)

# Save label mapping
label_dict = {int(i): str(label) for i, label in enumerate(label_mapping)}
with open("class_labels.json", "w") as f:
    json.dump(label_dict, f, indent=4)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Scale features (IMPORTANT for trees consistency)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

np.save("scaler_mean.npy", scaler.mean_)
np.save("scaler_std.npy", scaler.scale_)

NUM_FEATURES = X_train.shape[1]
NUM_CLASSES = len(np.unique(y))



In [10]:
print(scaler.mean_)
print(scaler.scale_)

[ 1.37339971e+01  2.35576923e-01  2.54807692e-01  2.62019231e-01
  3.41346154e-01  5.62500000e-01 -8.52624131e+00  9.73755196e-01
  1.49136893e-01  8.88272534e+00 -9.03941574e+00  5.21638285e+00
  1.38543630e+01  2.47596154e-01  2.54807692e-01  2.78846154e-01
  3.50961538e-01  5.93750000e-01 -8.36532333e+00  1.07270594e+00
  2.09302721e-01  1.15901167e+01 -1.09221227e+01  9.25055050e+00
  1.39745033e+01  2.78846154e-01  2.59615385e-01  3.05288462e-01
  3.77403846e-01  5.88942308e-01 -8.86148134e+00  8.85532318e-01
  4.48958854e-01  1.81437537e+01 -1.61066317e+01  1.62879294e+01
  1.40946123e+01  3.46153846e-01  2.74038462e-01  3.22115385e-01
  4.18269231e-01  6.68269231e-01 -9.06118999e+00  9.99239631e-01
  3.76269510e-01  2.38353641e+01 -3.90071014e+01  3.65941904e+01
  1.42147210e+01  4.06250000e-01  2.83653846e-01  3.46153846e-01
  5.52884615e-01  7.33173077e-01 -7.73897486e+00  1.74915999e+00
  6.26693795e-02  1.70100741e+01 -3.67929389e+01  3.25140561e+01
  1.43348338e+01  4.85576

In [11]:

# ============================================================
# TRAIN RANDOM FOREST
# ============================================================
rf = RandomForestClassifier(
    n_estimators=100,          # training size (can be large)
    max_depth=8,
    min_samples_leaf=15,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest Test Accuracy: {accuracy:.4f}")


# ============================================================
# C++ EXPORT FUNCTIONS
# ============================================================
def export_tree_cpp(tree, tree_id):
    t = tree.tree_
    lines = []

    lines.append(f"int tree_{tree_id}(float x[]) {{")

    def recurse(node, depth):
        indent = "  " * depth
        if t.feature[node] != _tree.TREE_UNDEFINED:
            f = t.feature[node]
            th = t.threshold[node]
            lines.append(f"{indent}if (x[{f}] <= {th:.6f}) {{")
            recurse(t.children_left[node], depth + 1)
            lines.append(f"{indent}}} else {{")
            recurse(t.children_right[node], depth + 1)
            lines.append(f"{indent}}}")
        else:
            cls = int(t.value[node][0].argmax())
            lines.append(f"{indent}return {cls};")

    recurse(0, 1)
    lines.append("}")
    return "\n".join(lines)


# ============================================================
# EXPORT FULL RANDOM FOREST (CONTROLLED SIZE)
# ============================================================
trees_to_export = rf.estimators_[:EXPORT_NUM_TREES]

cpp = []
cpp.append("// ====================================================")
cpp.append("// Auto-generated Random Forest (sklearn → C++)")
cpp.append("// ====================================================")
cpp.append("#include <stdint.h>\n")

# Export scaler
cpp.append(f"#define NUM_FEATURES {NUM_FEATURES}")
cpp.append(f"#define NUM_CLASSES {NUM_CLASSES}\n")

cpp.append("const float SCALER_MEAN[NUM_FEATURES] = {")
cpp.append(", ".join(f"{v:.6f}" for v in scaler.mean_))
cpp.append("};\n")

cpp.append("const float SCALER_STD[NUM_FEATURES] = {")
cpp.append(", ".join(f"{v:.6f}" for v in scaler.scale_))
cpp.append("};\n")

cpp.append("void scale(float x[]) {")
cpp.append("  for (int i = 0; i < NUM_FEATURES; i++) {")
cpp.append("    x[i] = (x[i] - SCALER_MEAN[i]) / SCALER_STD[i];")
cpp.append("  }")
cpp.append("}\n")

# Export trees
for i, tree in enumerate(trees_to_export):
    cpp.append(export_tree_cpp(tree, i))
    cpp.append("")

# Voting classifier
cpp.append("int classify(float x[]) {")
cpp.append("  int votes[NUM_CLASSES] = {0};")

for i in range(len(trees_to_export)):
    cpp.append(f"  votes[tree_{i}(x)]++;")

cpp.append("  int best = 0;")
cpp.append("  for (int i = 1; i < NUM_CLASSES; i++) {")
cpp.append("    if (votes[i] > votes[best]) best = i;")
cpp.append("  }")
cpp.append("  return best;")
cpp.append("}")

# Write file
with open(CPP_OUTPUT_FILE, "w") as f:
    f.write("\n".join(cpp))

print(f"[OK] Exported {len(trees_to_export)} trees to {CPP_OUTPUT_FILE}")


Random Forest Test Accuracy: 1.0000
[OK] Exported 7 trees to random_forest_model.cpp
